# Covariates, Comorbidity, additional analyses

In [ ]:
import numpy as np
import pandas as pd

import seaborn as sns
import pylab as plt

from scipy import stats
from statannot import add_stat_annotation
import joblib
import pingouin as pg

from importlib import reload
import sys
import os
sys.path.insert(1,'../')
import plots

In [ ]:
reload(plots)
image_path = plots.set_dir()
colors = plots.set_colors()
data_path = '/scratch/c.c21013066/data/ukbiobank/sample/withGP/noOsteo/'
acc_path = '/scratch/c.c21013066/data/ukbiobank/phenotypes/accelerometer/'

In [ ]:
dfs_new = pd.read_csv(f'{acc_path}unmatched_all_HCnoOsteo.csv',index_col=0)
subset = ['AllCauseDementia','AlzheimerDisease','AllCauseParkinsonism','ParkinsonDisease','Dystonia','Osteoarthritis','Depression']

In [ ]:
dfs_new.groupby('diagnosis')['Status'].value_counts()

# Comorbidity

In [ ]:
cooccurances = pd.DataFrame(columns=subset,index=subset)
for disease in subset:
    for codisease in subset:
        d1 = set(dfs_new.loc[np.logical_and(dfs_new['diagnosis']==disease,dfs_new['Status'].isin(['Prodromal','Diseased']))].index.values)
        d2 = set(dfs_new.loc[np.logical_and(dfs_new['diagnosis']==codisease,dfs_new['Status'].isin(['Prodromal','Diseased']))].index.values)
        cooccurances.loc[disease,codisease] = len(d1.intersection(d2))

In [ ]:
norm = (cooccurances / np.diag(cooccurances))*100
fig = plt.figure(figsize=(10,8))
plots.plot_context()
sns.heatmap(norm.astype(float),cbar_kws={'label': 'overlap [%]'},annot=True,fmt='.1f')
plt.savefig(f'{image_path}/overlap_disorders_HCnoOsteo.pdf',bbox_inches='tight',dpi=300)
plt.savefig(f'{image_path}/overlap_disorders_HCnoOsteo.png',bbox_inches='tight',dpi=300)

# Risk and Blood

In [ ]:
drop_healthy='nonHC'
name = 'ParkinsonDisease'

HC_clean = pd.read_csv('/scratch/c.c21013066/data/ukbiobank/phenotypes/accelerometer/allHCnoOsteo.csv')
#hc_res = pd.read_csv(f'{acc_path}HCnoOsteo_residuals_bmi.csv',index_col=0)
#HC_clean = pd.merge(hc,hc_res[['average acceleration residual']],right_index=True,left_index=True,how='outer')

In [ ]:
# check for correlation
y = 'No_wear_time_bias_adjusted_average_acceleration'
covs = np.hstack([['TownsendDeprivationIndex','BMI','Waist_Circumference','Hip_Circumference','Diastolic_BloodPressure','PulseRate',
         'BodyFat_Percentage','accelerometry_age'],HC_clean.columns[48:74]])
pearson = pd.DataFrame(columns=['n','r','CI95%','p-value','BF10','power'],index=covs)
for x in covs:
    nona = HC_clean.dropna(subset=[y,x])
    if x=='BMI' or x=='accelerometry_age':
        print(x,nona.shape[0])
        print(stats.pearsonr(nona[y], nona[x]))
    #pearson.loc[x,:] = stats.pearsonr(nona[y], nona[x])
    pearson.loc[x,:] = pg.corr(nona[y], nona[x]).values
pearson['sign'] = pearson[['p-value']]<0.05/pearson.shape[0]
pearson['sign'] = pearson['sign'].replace([True,False],['*',''])

In [ ]:
pearson.loc[['BMI','accelerometry_age'],:]

In [ ]:
fig = plt.figure(figsize=(20,2))
plots.plot_context()
ax = sns.heatmap(pearson[['r']].astype(float).T,annot=pearson[['sign']].T,fmt='',cmap='coolwarm',center=0,xticklabels=True,cbar_kws={'label': 'pearson r'})
ax.set_ylabel('average acceleration residual')
ax.set_yticklabels('')
plt.savefig(f'{image_path}/heatmap_riskblood_healthyNoOsteoresid.pdf',bbox_inches='tight',dpi=200)

In [ ]:
covs = ['male','AlcoholStatus_Current', 'AlcoholStatus_Previous',
       'SmokeStatus_Current', 'SmokeStatus_Previous',
       'DaytimeSleepiness_Often', 'AlcoholFrequency_LessThanWeekly',
       'family_Stroke', 'family_Diabetes', 'family_Severedepression',
       'family_Alzheimersdiseasedementia', 'family_Parkinsonsdisease']
mwu = pd.DataFrame(columns=['U-val','alternative','p-value','RBC','CLES','N_cases','N_controls','median_cases','median_controls'],index=covs)
for x in covs:
    print(x)
    cases = HC_clean.loc[HC_clean[x]==1,'No_wear_time_bias_adjusted_average_acceleration'].dropna()
    controls = HC_clean.loc[HC_clean[x]==0,'No_wear_time_bias_adjusted_average_acceleration'].dropna()
    print(cases.shape[0] + controls.shape[0])
    print(stats.mannwhitneyu(cases,controls))
    mwu.loc[x,['U-val','alternative','p-value','RBC','CLES']] = pg.mwu(cases,controls).values
    mwu.loc[x,'N_cases'] = cases.shape[0]
    mwu.loc[x,'N_controls'] = controls.shape[0]
    mwu.loc[x,'median_cases'] = cases.mean()
    mwu.loc[x,'median_controls'] = controls.mean()

In [ ]:
mwu

# PRS

In [ ]:
import yaml
def read_traits_file(input_path: str):
    with open(input_path, 'r') as f:
        traits_data = yaml.load(f, Loader=yaml.BaseLoader)
    return traits_data
traits = read_traits_file('/scratch/c.c21013066/Paper/ProdromalUKBB/resources/genetics/traits.yaml')
traits = pd.DataFrame(traits)

In [ ]:
path = '/scratch/c.c21013066/data/ukbiobank/'
score1 = pd.read_csv(f'{path}ukb52375.csv').set_index('eid')
# rename PRS in score file
score1.columns = score1.columns.str.replace('-0.0','')
PRSs = score1[traits.columns]
PRSs.columns = traits.loc['full_name',PRSs.columns]
for trait in PRSs.columns:
    PRSs[f'decile_{trait}_PRS'] = pd.qcut(PRSs[trait], 100, labels=False)
    PRSs[f'{trait}_PRS_group'] = PRSs[f'decile_{trait}_PRS'].replace(np.arange(100),
                                                           np.hstack([np.repeat('<3%',3),np.repeat('3-10%',7),
                                                                      np.repeat('10-90%',80),np.repeat('90-97%',7),
                                                                      np.repeat('>97%',3)]))
    PRSs[f'{trait}_PRS_group2'] = PRSs[f'decile_{trait}_PRS'].replace(np.arange(100),
                                                           np.hstack([np.repeat('<1%',1),np.repeat('1-5%',4),
                                                                      np.repeat('5-95%',90),np.repeat('95-99%',4),
                                                                      np.repeat('>99%',1)]))

In [ ]:
# load variant carriers
lrrk2 = pd.read_csv('/scratch/c.c21013066/data/ukbiobank/genetics/variant_WES/LRRK2_carriers.csv',index_col=0)
gba = pd.read_csv('/scratch/c.c21013066/data/ukbiobank/genetics/variant_WES/GBA_carriers.csv',index_col=0)
lr_vars = lrrk2.columns
gba_vars = gba.columns
lrrk2['any_variant'] = (lrrk2[lr_vars].sum(axis=1)>0).astype(int)
lrrk2['n_variants'] = (lrrk2[lr_vars]>0).sum(axis=1)
lrrk2['n_variants_additive'] = lrrk2[lr_vars].sum(axis=1)
gba['any_variant'] = (gba[gba_vars].sum(axis=1)>0).astype(int)
gba['n_variants'] = (gba[gba_vars]>0).sum(axis=1)
gba['n_variants_additive'] = gba[gba_vars].sum(axis=1)

variants = pd.merge(gba,lrrk2,right_index=True,left_index=True,how='outer',suffixes=['','_LRRK2'])
var_cols = variants.filter(regex='variants').columns

In [ ]:
match = pd.read_csv(f'{acc_path}/matched_all_residual_BMI_HCnoOsteo.csv').set_index('eid')
hc_res = pd.read_csv(f'{acc_path}HCnoOsteo_residuals_bmi.csv',index_col=0)
hc = pd.merge(HC_clean.set_index('eid'),hc_res.filter(regex='residual'),right_index=True,left_index=True)
alle = pd.concat([match,hc])
alle = alle.drop_duplicates(keep='first')
alle = pd.merge(alle,PRSs,right_index=True,left_index=True,how='outer')

In [ ]:
for y,kind in zip(['average acceleration residual_bmi'],['resbmi']):
    rvalues = pd.DataFrame(index=traits.loc['full_name'],
                       columns=['r','p'])
    for disease in traits.loc['full_name']:
        nona = alle.dropna(subset=[y,disease])# or only hc
        print(nona.shape)
        rv, pv = stats.pearsonr(nona[disease],nona[y])
        rvalues.loc[disease,'r'] = rv
        rvalues.loc[disease,'p'] = pv
        rvalues.loc[disease,'s'] = rvalues.loc[disease,'p'] < (0.05/rvalues.shape[0])
    rvalues['s'] = rvalues['s'].replace([True,False],['*',''])
    plots.plot_context()
    g = sns.clustermap(rvalues[['r']].astype(float).T,annot=rvalues[['s']].T,cmap='coolwarm',center=0,fmt='',
                  figsize=(15,5),cbar_kws={'label': 'pearson r'},row_cluster=False,method='ward',xticklabels=True)
    ax = g.ax_heatmap
    ax.set_xlabel('');
    ax.set_ylabel(y)
    ax.set_yticklabels('')
    #plt.savefig(f'{image_path}/PRS_clustermap_acc{kind}_pearson.pdf',bbox_inches='tight',dpi=200)